# 00.3 · The three kinds of learning

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/00-big-picture/03-three-kinds-of-learning/lesson.ipynb)

Read the [README](README.md) first. Here you run one example of each kind on the same 20 ShopWave customers and watch the *shape* of the code change:

- supervised: `fit(X, y)`
- unsupervised: `fit(X)` with no `y`
- reinforcement: a loop of act → reward → update

## The data

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd
cust = pd.read_csv(data_path("shopwave_customers_small.csv"))
cust

,customer_id,visits_last_month,items_last_month,days_since_last,country,spend_next_month,churned
0,C001,2,2,55,UK,27.61,1
1,C002,11,8,44,FR,220.42,0
2,C003,10,6,22,DE,176.21,0
3,C004,7,6,58,UK,127.58,1
4,C005,7,5,25,FR,150.95,1
5,C006,13,7,20,FR,250.03,0
6,C007,2,2,54,UK,47.33,1
7,C008,10,5,22,UK,196.92,0
8,C009,3,3,5,DE,94.82,1
9,C010,2,1,28,UK,45.05,1


## Supervised: the answers exist

`churned` is known for all 20 customers. We give the model features **and** answers.

In [2]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["visits_last_month", "items_last_month", "days_since_last"]
X = cust[features]
y = cust["churned"]                       # <-- the answer column

model = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
print(f"accuracy on these 20: {(model.predict(X) == y).mean():.0%}")
print(export_text(model, feature_names=features))

accuracy on these 20: 85%
|--- visits_last_month <= 9.00
|   |--- visits_last_month <= 5.00
|   |   |--- class: 1
|   |--- visits_last_month >  5.00
|   |   |--- class: 1
|--- visits_last_month >  9.00
|   |--- days_since_last <= 31.50
|   |   |--- class: 0
|   |--- days_since_last >  31.50
|   |   |--- class: 0



(Scoring on the rows it learned from flatters it; Module 04 fixes that. The shape is what matters: `fit(X, y)`.)

## Unsupervised: no answers

Same features, no `y`. K-Means is asked for 3 groups and finds them by putting similar customers together.

In [3]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)          # put columns on the same scale first
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_scaled)   # <-- no y
cust["segment"] = km.labels_

cust.groupby("segment")[features + ["spend_next_month"]].mean().round(1)

,visits_last_month,items_last_month,days_since_last,spend_next_month
segment,,,,
0,5.2,4.0,50.8,99.0
1,10.8,6.6,25.4,206.4
2,2.3,2.0,15.7,73.0


Read each row of the summary as a persona: look at visits, days since last order and spend. Nobody told the algorithm what the groups mean; you interpret them. (Module 06 does this properly.)

## Reinforcement: learn from rewards

No table at all. An agent chooses a discount for each order, sees the profit, and updates its beliefs. Watch it discover the best level.

In [4]:
import numpy as np
rng = np.random.default_rng(0)

# Three discount levels. The TRUE average profit per order for each is hidden
# from the agent; it only ever sees the profit of the orders it actually tries.
true_profit = {"0%": 6.0, "5%": 8.5, "15%": 7.0}
discounts = list(true_profit)

estimates = {d: 0.0 for d in discounts}   # what the agent believes so far
tries = {d: 0 for d in discounts}
total_profit = 0

for order in range(300):
    # explore 10% of the time, otherwise exploit the best estimate so far
    if rng.random() < 0.10 or order < 3:
        d = discounts[order % 3] if order < 3 else rng.choice(discounts)
    else:
        d = max(estimates, key=estimates.get)
    reward = true_profit[d] + rng.normal(0, 3)          # noisy profit from this one order
    tries[d] += 1
    estimates[d] += (reward - estimates[d]) / tries[d]  # running average
    total_profit += reward

print("tries per discount :", tries)
print("learned estimates  :", {d: round(v, 2) for d, v in estimates.items()})
print("true values        :", true_profit)
print("agent's final choice:", max(estimates, key=estimates.get))

tries per discount : {'0%': 10, '5%': 275, '15%': 15}
learned estimates  : {'0%': 5.63, '5%': 8.4, '15%': 6.55}
true values        : {'0%': 6.0, '5%': 8.5, '15%': 7.0}
agent's final choice: 5%


The agent never saw a label saying "5% is best". It found out by trying, mostly exploiting what looked good and occasionally exploring. Notice the cost: it spent real orders on the worse discounts while learning.

## On ShopWave's real columns

From invoice, product, quantity, date, price, customer, country you can **build** supervised targets:

- churn: no order in the next 90 days (classification)
- next-quarter spend (regression)
- will this invoice contain a return? (classification)

and ask unsupervised questions with no target:

- customer segments, products bought together, unusual invoices.

Lesson 00.5 does both.

## Quiz answers

1. *Do we already have the answers (labels) for past examples?*
2. Delivery days → **regression** (a number). Late/on-time → **classification** (a category).
3. Labels often do not exist and have to be created by people, one row at a time, consistently. Photos must be tagged; fraud must be confirmed; churn must be defined.
4. No. K-Means returns exactly as many clusters as you ask for. Whether they are meaningful is for you to judge by profiling them and checking they are stable.
5. It learns by trying actions in the real world, which costs money or risk; it needs a simulator or cheap mistakes; and most business questions already have labels, so supervised learning is simpler and safer.

**Next:** [00.4 The 7-step ML workflow](../04-ml-workflow/)